# Hypothesis 4: concatenating the previous representation matters

The full update projects `Cₜ = [Xₜ, Zₜ]`. We compare it with a replacement update that discards `Xₜ` and retains only the newly projected tree representation `Zₜ`. Both outputs have the same final dimension.

In [1]:
import warnings
from sklearn.random_projection import DataDimensionalityWarning
warnings.filterwarnings(
    "ignore",
    category=DataDimensionalityWarning,
    message="The number of components is higher than the number of features.*",
)

import sys
from pathlib import Path
notebooks_dir = Path.cwd() / 'notebooks'
if not (notebooks_dir / '_hypothesis_utils.py').exists(): notebooks_dir = Path.cwd()
sys.path.insert(0, str(notebooks_dir))
import pandas as pd
from _hypothesis_utils import CLASSIFICATION_SEEDS, NoConcatForestSketch, classification_data, classification_score, downstream_classifier, forest_classifier, forest_sketch

In [2]:
rows = []
for seed in CLASSIFICATION_SEEDS[:2]:
    X_train, _, X_test, y_train, _, y_test = classification_data(seed)
    dimension = 20 * X_train.shape[1]
    print(f'seed={seed}: p={X_train.shape[1]}, d=20*p={dimension}')
    for name, representation in [
        ('full concatenation', forest_sketch(seed, dimension, 2)),
        ('replacement without concatenation', NoConcatForestSketch(forest_classifier(seed, n_estimators=100), dimension, 2, seed)),
    ]:
        X_train_view = representation.fit_transform(X_train, y_train)
        X_test_view = representation.transform(X_test)
        accuracy, errors = classification_score(
            downstream_classifier(seed), X_train_view, y_train, X_test_view, y_test
        )
        rows.append({'seed': seed, 'method': name, 'accuracy': accuracy, 'errors': errors})
results = pd.DataFrame(rows)
summary = results.groupby('method').accuracy.agg(['mean', 'std']).round(3)
display(results.round(3))
display(summary)
gain = summary.loc['full concatenation', 'mean'] - summary.loc['replacement without concatenation', 'mean']
print(f'Hypothesis 4: {"SUPPORTED" if gain > 0 else "NOT SUPPORTED"} — concatenation accuracy gain={gain:.3f}')

seed=0: p=120, d=20*p=2400


seed=1: p=120, d=20*p=2400


,seed,method,accuracy,errors
0,0,full concatenation,0.809,84
1,0,replacement without concatenation,0.752,109
2,1,full concatenation,0.850,66
3,1,replacement without concatenation,0.811,83


,mean,std
method,,
full concatenation,0.830,0.029
replacement without concatenation,0.782,0.042


Hypothesis 4: SUPPORTED — concatenation accuracy gain=0.048


## Conclusion

**Hypothesis:** concatenating the previous representation with the new tree representation is better than replacing the previous representation.

**Experiment:** compare the full two-iteration update with a matched replacement ablation that retains only `Zₜ`, across two seeds at `d=8192`.

**Measure:** held-out accuracy and integer error count from the same downstream classifier.

**Result:** weakly supported in this run: concatenation improved mean accuracy by 0.001, so the effect is practically negligible and needs more seeds.